In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 🌡️ Análisis Climático Colombia — Modelo de Regresión Lineal\n",
    "**Proyecto:** ETL-Proyecto-API  \n",
    "**Objetivo:** Predecir la temperatura máxima diaria usando variables climáticas  \n",
    "**Técnica:** Regresión Lineal Simple y Múltiple  \n",
    "**Fuente de datos:** PostgreSQL (tabla `mediciones` + `ciudades`)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 0. Librerías e Importaciones"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "import pickle\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import matplotlib.ticker as mticker\n",
    "import seaborn as sns\n",
    "from dotenv import load_dotenv\n",
    "from sqlalchemy import create_engine, text\n",
    "from datetime import datetime, timedelta\n",
    "\n",
    "# Scikit-learn\n",
    "from sklearn.linear_model import LinearRegression\n",
    "from sklearn.model_selection import train_test_split, cross_val_score, KFold\n",
    "from sklearn.preprocessing import StandardScaler, PolynomialFeatures\n",
    "from sklearn.metrics import (\n",
    "    mean_squared_error, mean_absolute_error,\n",
    "    r2_score, mean_absolute_percentage_error\n",
    ")\n",
    "from sklearn.pipeline import Pipeline\n",
    "\n",
    "# Estilo global de gráficas\n",
    "plt.style.use('dark_background')\n",
    "sns.set_theme(style='darkgrid', palette='coolwarm')\n",
    "COLORS = ['#00d4ff', '#ff6b6b', '#ffd93d', '#6bcb77', '#c77dff', '#ff9f1c']\n",
    "\n",
    "print('✅ Librerías cargadas correctamente')\n",
    "print(f'📅 Fecha de ejecución: {datetime.now().strftime(\"%Y-%m-%d %H:%M:%S\")}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Conexión a la Base de Datos"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Carga variables de entorno desde el .env del proyecto\n",
    "load_dotenv(dotenv_path='../.env')\n",
    "\n",
    "DB_HOST = os.getenv('DB_HOST', 'localhost')\n",
    "DB_PORT = os.getenv('DB_PORT', '5432')\n",
    "DB_NAME = os.getenv('DB_NAME', 'clima_db')\n",
    "DB_USER = os.getenv('DB_USER', 'clima_user')\n",
    "DB_PASS = os.getenv('DB_PASSWORD', 'clima1234')\n",
    "\n",
    "DATABASE_URL = f'postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}'\n",
    "\n",
    "try:\n",
    "    engine = create_engine(DATABASE_URL)\n",
    "    with engine.connect() as conn:\n",
    "        conn.execute(text('SELECT 1'))\n",
    "    print(f'✅ Conectado a PostgreSQL: {DB_NAME}@{DB_HOST}:{DB_PORT}')\n",
    "except Exception as e:\n",
    "    print(f'❌ Error de conexión: {e}')\n",
    "    print('⚠️  Asegúrate de que Docker esté corriendo: docker compose up -d')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Carga y Exploración Inicial de Datos"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "QUERY = \"\"\"\n",
    "    SELECT\n",
    "        m.id,\n",
    "        c.nombre        AS ciudad,\n",
    "        c.departamento,\n",
    "        c.latitud,\n",
    "        c.longitud,\n",
    "        m.fecha_consulta,\n",
    "        m.temperatura,\n",
    "        m.temp_min,\n",
    "        m.temp_max,\n",
    "        m.sensacion_termica,\n",
    "        m.humedad,\n",
    "        m.presion,\n",
    "        m.velocidad_viento,\n",
    "        m.direccion_viento,\n",
    "        m.nubosidad,\n",
    "        m.descripcion\n",
    "    FROM mediciones m\n",
    "    JOIN ciudades c ON m.ciudad_id = c.id\n",
    "    ORDER BY m.fecha_consulta DESC;\n",
    "\"\"\"\n",
    "\n",
    "df_raw = pd.read_sql(QUERY, engine)\n",
    "print(f'📦 Registros cargados: {len(df_raw):,}')\n",
    "print(f'🏙️  Ciudades únicas: {df_raw[\"ciudad\"].nunique()}')\n",
    "print(f'📅 Rango de fechas: {df_raw[\"fecha_consulta\"].min()} → {df_raw[\"fecha_consulta\"].max()}')\n",
    "df_raw.head()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Info general del DataFrame\n",
    "print('=== TIPOS DE DATOS ===')\n",
    "print(df_raw.dtypes)\n",
    "print('\\n=== VALORES NULOS ===')\n",
    "print(df_raw.isnull().sum())\n",
    "print('\\n=== ESTADÍSTICAS DESCRIPTIVAS ===')\n",
    "df_raw.describe().round(2)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Limpieza y Preprocesamiento"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "df = df_raw.copy()\n",
    "\n",
    "# Convertir fecha\n",
    "df['fecha_consulta'] = pd.to_datetime(df['fecha_consulta'])\n",
    "\n",
    "# Extraer features temporales\n",
    "df['hora']       = df['fecha_consulta'].dt.hour\n",
    "df['dia']        = df['fecha_consulta'].dt.day\n",
    "df['mes']        = df['fecha_consulta'].dt.month\n",
    "df['dia_semana'] = df['fecha_consulta'].dt.dayofweek  # 0=Lunes\n",
    "df['fecha_solo'] = df['fecha_consulta'].dt.date\n",
    "\n",
    "# Eliminar duplicados\n",
    "antes = len(df)\n",
    "df.drop_duplicates(subset=['ciudad', 'fecha_consulta'], inplace=True)\n",
    "print(f'🗑️  Duplicados eliminados: {antes - len(df)}')\n",
    "\n",
    "# Eliminar filas con nulos en columnas clave\n",
    "cols_clave = ['temperatura', 'temp_max', 'temp_min', 'humedad',\n",
    "              'presion', 'velocidad_viento', 'nubosidad']\n",
    "df.dropna(subset=cols_clave, inplace=True)\n",
    "\n",
    "# Eliminar outliers extremos con IQR en temperatura\n",
    "Q1  = df['temperatura'].quantile(0.25)\n",
    "Q3  = df['temperatura'].quantile(0.75)\n",
    "IQR = Q3 - Q1\n",
    "df  = df[(df['temperatura'] >= Q1 - 3*IQR) & (df['temperatura'] <= Q3 + 3*IQR)]\n",
    "\n",
    "# Feature engineering: rango térmico diario\n",
    "df['rango_termico'] = df['temp_max'] - df['temp_min']\n",
    "\n",
    "# Latitud absoluta (proxy geográfico)\n",
    "df['abs_lat'] = df['latitud'].abs()\n",
    "\n",
    "print(f'✅ Dataset limpio: {len(df):,} registros')\n",
    "print(f'🏙️  Ciudades: {df[\"ciudad\"].nunique()}')\n",
    "df[cols_clave].describe().round(2)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Análisis Exploratorio de Datos (EDA)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# --- Distribución de temperatura por ciudad ---\n",
    "top_ciudades = df['ciudad'].value_counts().head(15).index\n",
    "df_top = df[df['ciudad'].isin(top_ciudades)]\n",
    "\n",
    "fig, axes = plt.subplots(1, 2, figsize=(18, 7))\n",
    "fig.suptitle('Distribución de Temperatura — Top 15 Ciudades', fontsize=16, fontweight='bold', color='white')\n",
    "\n",
    "order = df_top.groupby('ciudad')['temperatura'].median().sort_values(ascending=False).index\n",
    "\n",
    "# Boxplot\n",
    "sns.boxplot(data=df_top, x='temperatura', y='ciudad', order=order,\n",
    "            palette='coolwarm', ax=axes[0])\n",
    "axes[0].set_title('Boxplot de Temperatura', color='white')\n",
    "axes[0].set_xlabel('Temperatura (°C)', color='white')\n",
    "axes[0].set_ylabel('Ciudad', color='white')\n",
    "axes[0].axvline(df['temperatura'].mean(), color='yellow', linestyle='--', alpha=0.7, label='Promedio nacional')\n",
    "axes[0].legend()\n",
    "\n",
    "# Violin\n",
    "sns.violinplot(data=df_top, x='temperatura', y='ciudad', order=order,\n",
    "               palette='coolwarm', ax=axes[1], inner='quartile')\n",
    "axes[1].set_title('Distribución (Violin)', color='white')\n",
    "axes[1].set_xlabel('Temperatura (°C)', color='white')\n",
    "axes[1].set_ylabel('')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('../data/eda_distribucion_temperatura.png', dpi=120, bbox_inches='tight', facecolor='#1a1a2e')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# --- Evolución temporal promedio nacional ---\n",
    "ts = df.groupby('fecha_consulta').agg(\n",
    "    temp_prom=('temperatura', 'mean'),\n",
    "    humedad_prom=('humedad', 'mean'),\n",
    "    viento_prom=('velocidad_viento', 'mean')\n",
    ").reset_index()\n",
    "\n",
    "fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)\n",
    "fig.suptitle('Evolución Temporal — Variables Climáticas Colombia', fontsize=15, fontweight='bold', color='white')\n",
    "\n",
    "axes[0].plot(ts['fecha_consulta'], ts['temp_prom'], color=COLORS[0], linewidth=1.5)\n",
    "axes[0].fill_between(ts['fecha_consulta'], ts['temp_prom'], alpha=0.2, color=COLORS[0])\n",
    "axes[0].set_ylabel('Temp. Promedio (°C)', color='white')\n",
    "axes[0].set_title('Temperatura', color='#aaa', fontsize=11)\n",
    "\n",
    "axes[1].plot(ts['fecha_consulta'], ts['humedad_prom'], color=COLORS[1], linewidth=1.5)\n",
    "axes[1].fill_between(ts['fecha_consulta'], ts['humedad_prom'], alpha=0.2, color=COLORS[1])\n",
    "axes[1].set_ylabel('Humedad (%)', color='white')\n",
    "axes[1].set_title('Humedad', color='#aaa', fontsize=11)\n",
    "\n",
    "axes[2].plot(ts['fecha_consulta'], ts['viento_prom'], color=COLORS[2], linewidth=1.5)\n",
    "axes[2].fill_between(ts['fecha_consulta'], ts['viento_prom'], alpha=0.2, color=COLORS[2])\n",
    "axes[2].set_ylabel('Viento (m/s)', color='white')\n",
    "axes[2].set_title('Velocidad del Viento', color='#aaa', fontsize=11)\n",
    "axes[2].set_xlabel('Fecha', color='white')\n",
    "\n",
    "for ax in axes:\n",
    "    ax.grid(True, alpha=0.2)\n",
    "    ax.tick_params(colors='white')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('../data/eda_serie_temporal.png', dpi=120, bbox_inches='tight', facecolor='#1a1a2e')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# --- Matriz de correlación ---\n",
    "cols_corr = ['temperatura', 'temp_min', 'temp_max', 'sensacion_termica',\n",
    "             'humedad', 'presion', 'velocidad_viento', 'nubosidad',\n",
    "             'rango_termico', 'hora', 'latitud']\n",
    "\n",
    "corr_matrix = df[cols_corr].corr()\n",
    "\n",
    "fig, ax = plt.subplots(figsize=(12, 9))\n",
    "mask = np.triu(np.ones_like(corr_matrix, dtype=bool))\n",
    "sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',\n",
    "            cmap='coolwarm', center=0, vmin=-1, vmax=1,\n",
    "            linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})\n",
    "ax.set_title('Matriz de Correlación — Variables Climáticas', fontsize=14, fontweight='bold', color='white', pad=15)\n",
    "ax.tick_params(colors='white', labelsize=9)\n",
    "plt.tight_layout()\n",
    "plt.savefig('../data/eda_correlacion.png', dpi=120, bbox_inches='tight', facecolor='#1a1a2e')\n",
    "plt.show()\n",
    "\n",
    "print('\\n🔗 Correlación con temperatura (ordenado):')\n",
    "print(corr_matrix['temperatura'].drop('temperatura').sort_values(ascending=False).to_string())"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Preparación de Features para el Modelo"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "TARGET = 'temp_max'\n",
    "\n",
    "FEATURES_SIMPLE = ['temperatura']\n",
    "\n",
    "FEATURES_MULTIPLE = [\n",
    "    'temperatura',\n",
    "    'temp_min',\n",
    "    'sensacion_termica',\n",
    "    'humedad',\n",
    "    'presion',\n",
    "    'velocidad_viento',\n",
    "    'nubosidad',\n",
    "    'rango_termico',\n",
    "    'hora',\n",
    "    'latitud',\n",
    "]\n",
    "\n",
    "df_model = df[FEATURES_MULTIPLE + [TARGET]].dropna()\n",
    "print(f'📊 Dataset para modelado: {len(df_model):,} registros')\n",
    "print(f'🎯 Variable objetivo: {TARGET}')\n",
    "print(f'🔢 Features: {len(FEATURES_MULTIPLE)}')\n",
    "\n",
    "X = df_model[FEATURES_MULTIPLE].values\n",
    "y = df_model[TARGET].values\n",
    "\n",
    "# Split 80/20\n",
    "X_train, X_test, y_train, y_test = train_test_split(\n",
    "    X, y, test_size=0.2, random_state=42\n",
    ")\n",
    "\n",
    "print(f'\\n📐 Entrenamiento: {X_train.shape[0]:,} | Prueba: {X_test.shape[0]:,}')\n",
    "\n",
    "# Escalado — fit solo en train para evitar data leakage\n",
    "scaler = StandardScaler()\n",
    "X_train_sc = scaler.fit_transform(X_train)\n",
    "X_test_sc  = scaler.transform(X_test)\n",
    "\n",
    "print('✅ Split y escalado completados')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Modelo 1: Regresión Lineal Simple (Temperatura → Temp. Máxima)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "X_simple = df_model[['temperatura']].values\n",
    "y_simple = df_model[TARGET].values\n",
    "\n",
    "X_s_train, X_s_test, y_s_train, y_s_test = train_test_split(\n",
    "    X_simple, y_simple, test_size=0.2, random_state=42\n",
    ")\n",
    "\n",
    "lr_simple = LinearRegression()\n",
    "lr_simple.fit(X_s_train, y_s_train)\n",
    "y_pred_simple = lr_simple.predict(X_s_test)\n",
    "\n",
    "r2_s   = r2_score(y_s_test, y_pred_simple)\n",
    "mae_s  = mean_absolute_error(y_s_test, y_pred_simple)\n",
    "rmse_s = np.sqrt(mean_squared_error(y_s_test, y_pred_simple))\n",
    "\n",
    "print('=== REGRESIÓN LINEAL SIMPLE ===')\n",
    "print(f'📐 Ecuación: temp_max = {lr_simple.intercept_:.4f} + {lr_simple.coef_[0]:.4f} × temperatura')\n",
    "print(f'📈 R²   : {r2_s:.4f}  ({r2_s*100:.1f}% varianza explicada)')\n",
    "print(f'📉 MAE  : {mae_s:.4f} °C')\n",
    "print(f'📉 RMSE : {rmse_s:.4f} °C')\n",
    "\n",
    "fig, axes = plt.subplots(1, 2, figsize=(16, 6))\n",
    "fig.suptitle('Regresión Lineal Simple: Temperatura → Temperatura Máxima', fontsize=14, fontweight='bold', color='white')\n",
    "\n",
    "# Scatter + línea\n",
    "axes[0].scatter(X_s_test, y_s_test, alpha=0.4, color=COLORS[0], s=15, label='Datos reales')\n",
    "x_line = np.linspace(X_s_test.min(), X_s_test.max(), 100).reshape(-1, 1)\n",
    "axes[0].plot(x_line, lr_simple.predict(x_line), color=COLORS[1], linewidth=2.5, label='Línea de regresión')\n",
    "axes[0].set_xlabel('Temperatura Actual (°C)', color='white')\n",
    "axes[0].set_ylabel('Temperatura Máxima (°C)', color='white')\n",
    "axes[0].set_title(f'Scatter + Regresión  |  R² = {r2_s:.3f}', color='white')\n",
    "axes[0].legend()\n",
    "axes[0].grid(True, alpha=0.2)\n",
    "\n",
    "# Residuos\n",
    "residuos_s = y_s_test - y_pred_simple\n",
    "axes[1].scatter(y_pred_simple, residuos_s, alpha=0.4, color=COLORS[2], s=15)\n",
    "axes[1].axhline(0, color=COLORS[1], linewidth=2, linestyle='--')\n",
    "axes[1].set_xlabel('Valores Predichos (°C)', color='white')\n",
    "axes[1].set_ylabel('Residuos (°C)', color='white')\n",
    "axes[1].set_title('Gráfica de Residuos', color='white')\n",
    "axes[1].grid(True, alpha=0.2)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('../data/ml_regresion_simple.png', dpi=120, bbox_inches='tight', facecolor='#1a1a2e')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Modelo 2: Regresión Lineal Múltiple"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "lr_multi = LinearRegression()\n",
    "lr_multi.fit(X_train_sc, y_train)\n",
    "y_pred_multi = lr_multi.predict(X_test_sc)\n",
    "\n",
    "r2_m   = r2_score(y_test, y_pred_multi)\n",
    "mae_m  = mean_absolute_error(y_test, y_pred_multi)\n",
    "rmse_m = np.sqrt(mean_squared_error(y_test, y_pred_multi))\n",
    "mape_m = mean_absolute_percentage_error(y_test, y_pred_multi) * 100\n",
    "\n",
    "print('=== REGRESIÓN LINEAL MÚLTIPLE ===')\n",
    "print(f'📈 R²   : {r2_m:.4f}  ({r2_m*100:.1f}% varianza explicada)')\n",
    "print(f'📉 MAE  : {mae_m:.4f} °C')\n",
    "print(f'📉 RMSE : {rmse_m:.4f} °C')\n",
    "print(f'📉 MAPE : {mape_m:.2f}%')\n",
    "\n",
    "coef_df = pd.DataFrame({\n",
    "    'Feature': FEATURES_MULTIPLE,\n",
    "    'Coeficiente': lr_multi.coef_\n",
    "}).sort_values('Coeficiente', ascending=False)\n",
    "\n",
    "print('\\n📊 Coeficientes del modelo (estandarizados):')\n",
    "print(coef_df.to_string(index=False))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, axes = plt.subplots(2, 2, figsize=(16, 12))\n",
    "fig.suptitle('Regresión Lineal Múltiple — Análisis Completo', fontsize=15, fontweight='bold', color='white')\n",
    "\n",
    "# 1. Real vs Predicho\n",
    "axes[0,0].scatter(y_test, y_pred_multi, alpha=0.4, color=COLORS[0], s=15)\n",
    "min_val = min(y_test.min(), y_pred_multi.min())\n",
    "max_val = max(y_test.max(), y_pred_multi.max())\n",
    "axes[0,0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Predicción perfecta')\n",
    "axes[0,0].set_xlabel('Valores Reales (°C)', color='white')\n",
    "axes[0,0].set_ylabel('Valores Predichos (°C)', color='white')\n",
    "axes[0,0].set_title(f'Real vs Predicho  |  R² = {r2_m:.3f}', color='white')\n",
    "axes[0,0].legend()\n",
    "axes[0,0].grid(True, alpha=0.2)\n",
    "\n",
    "# 2. Distribución de residuos\n",
    "residuos_m = y_test - y_pred_multi\n",
    "axes[0,1].hist(residuos_m, bins=40, color=COLORS[1], alpha=0.8, edgecolor='white', linewidth=0.3)\n",
    "axes[0,1].axvline(0, color='yellow', linewidth=2, linestyle='--', label='Error cero')\n",
    "axes[0,1].axvline(residuos_m.mean(), color=COLORS[2], linewidth=2, linestyle='-.', label=f'Media: {residuos_m.mean():.3f}')\n",
    "axes[0,1].set_xlabel('Residuo (°C)', color='white')\n",
    "axes[0,1].set_ylabel('Frecuencia', color='white')\n",
    "axes[0,1].set_title('Distribución de Residuos', color='white')\n",
    "axes[0,1].legend()\n",
    "axes[0,1].grid(True, alpha=0.2)\n",
    "\n",
    "# 3. Importancia de features\n",
    "colors_bar = [COLORS[0] if c > 0 else COLORS[1] for c in coef_df['Coeficiente']]\n",
    "axes[1,0].barh(coef_df['Feature'], coef_df['Coeficiente'], color=colors_bar, alpha=0.85)\n",
    "axes[1,0].axvline(0, color='white', linewidth=0.8)\n",
    "axes[1,0].set_xlabel('Coeficiente (estandarizado)', color='white')\n",
    "axes[1,0].set_title('Importancia de Variables (Coeficientes)', color='white')\n",
    "axes[1,0].grid(True, alpha=0.2, axis='x')\n",
    "axes[1,0].tick_params(colors='white')\n",
    "\n",
    "# 4. Residuos vs Predichos (homocedasticidad)\n",
    "axes[1,1].scatter(y_pred_multi, residuos_m, alpha=0.4, color=COLORS[3], s=15)\n",
    "axes[1,1].axhline(0, color=COLORS[1], linewidth=2, linestyle='--')\n",
    "axes[1,1].axhline( residuos_m.std(), color='gray', linewidth=1, linestyle=':', label=f'+1σ = {residuos_m.std():.2f}')\n",
    "axes[1,1].axhline(-residuos_m.std(), color='gray', linewidth=1, linestyle=':', label='-1σ')\n",
    "axes[1,1].set_xlabel('Valores Predichos (°C)', color='white')\n",
    "axes[1,1].set_ylabel('Residuos (°C)', color='white')\n",
    "axes[1,1].set_title('Residuos vs Predichos (Homocedasticidad)', color='white')\n",
    "axes[1,1].legend()\n",
    "axes[1,1].grid(True, alpha=0.2)\n",
    "\n",
    "for row in axes:\n",
    "    for ax in row:\n",
    "        ax.tick_params(colors='white')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('../data/ml_regresion_multiple.png', dpi=120, bbox_inches='tight', facecolor='#1a1a2e')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 8. Modelo 3: Regresión Polinómica (grado 2)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Pipeline: PolynomialFeatures → StandardScaler → LinearRegression\n",
    "# El scaler va DENTRO del pipeline para evitar data leakage en cross-val\n",
    "pipeline_poly = Pipeline([\n",
    "    ('poly',   PolynomialFeatures(degree=2, include_bias=False)),\n",
    "    ('scaler', StandardScaler()),\n",
    "    ('model',  LinearRegression())\n",
    "])\n",
    "\n",
    "pipeline_poly.fit(X_train, y_train)\n",
    "y_pred_poly = pipeline_poly.predict(X_test)\n",
    "\n",
    "r2_p   = r2_score(y_test, y_pred_poly)\n",
    "mae_p  = mean_absolute_error(y_test, y_pred_poly)\n",
    "rmse_p = np.sqrt(mean_squared_error(y_test, y_pred_poly))\n",
    "\n",
    "print('=== REGRESIÓN POLINÓMICA (grado 2) ===')\n",
    "print(f'📈 R²   : {r2_p:.4f}  ({r2_p*100:.1f}% varianza explicada)')\n",
    "print(f'📉 MAE  : {mae_p:.4f} °C')\n",
    "print(f'📉 RMSE : {rmse_p:.4f} °C')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 9. Validación Cruzada 10-Fold\n",
    "\n",
    "> ⚠️ **Nota importante:** Para evitar *data leakage*, el `StandardScaler` se aplica **dentro** de cada fold usando `Pipeline`. Así el scaler aprende solo de los datos de entrenamiento de cada fold y no ve los datos de validación."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "kf = KFold(n_splits=10, shuffle=True, random_state=42)\n",
    "\n",
    "# ✅ Pipeline para modelo simple — scaler dentro del fold\n",
    "pipe_simple = Pipeline([\n",
    "    ('scaler', StandardScaler()),\n",
    "    ('model',  LinearRegression())\n",
    "])\n",
    "cv_simple = cross_val_score(pipe_simple, X_simple, y_simple, cv=kf, scoring='r2')\n",
    "\n",
    "# ✅ Pipeline para modelo múltiple — scaler dentro del fold\n",
    "pipe_multi = Pipeline([\n",
    "    ('scaler', StandardScaler()),\n",
    "    ('model',  LinearRegression())\n",
    "])\n",
    "cv_multi = cross_val_score(pipe_multi, X, y, cv=kf, scoring='r2')\n",
    "\n",
    "# ✅ Pipeline polinómico — ya tiene scaler interno\n",
    "cv_poly = cross_val_score(pipeline_poly, X, y, cv=kf, scoring='r2')\n",
    "\n",
    "print('=== VALIDACIÓN CRUZADA 10-FOLD ===')\n",
    "print(f'Regresión Simple    → R² prom: {cv_simple.mean():.4f} ± {cv_simple.std():.4f}')\n",
    "print(f'Regresión Múltiple  → R² prom: {cv_multi.mean():.4f}  ± {cv_multi.std():.4f}')\n",
    "print(f'Regresión Polinómica→ R² prom: {cv_poly.mean():.4f}   ± {cv_poly.std():.4f}')\n",
    "\n",
    "# Gráfica comparativa\n",
    "fig, ax = plt.subplots(figsize=(12, 5))\n",
    "ax.set_facecolor('#1a1a2e')\n",
    "fig.patch.set_facecolor('#1a1a2e')\n",
    "\n",
    "data_cv    = [cv_simple, cv_multi, cv_poly]\n",
    "labels_cv  = ['Lineal Simple', 'Lineal Múltiple', 'Polinómica (g=2)']\n",
    "bp = ax.boxplot(data_cv, patch_artist=True, labels=labels_cv, notch=True)\n",
    "\n",
    "for patch, color in zip(bp['boxes'], COLORS):\n",
    "    patch.set_facecolor(color)\n",
    "    patch.set_alpha(0.7)\n",
    "\n",
    "ax.set_ylabel('R² Score', color='white', fontsize=12)\n",
    "ax.set_title('Comparación de Modelos — Validación Cruzada 10-Fold', color='white', fontsize=14, fontweight='bold')\n",
    "ax.tick_params(colors='white')\n",
    "ax.grid(True, alpha=0.2)\n",
    "plt.tight_layout()\n",
    "plt.savefig('../data/ml_cross_validation.png', dpi=120, bbox_inches='tight', facecolor='#1a1a2e')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 10. Resumen de Métricas y Comparación de Modelos"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "resultados = pd.DataFrame({\n",
    "    'Modelo':        ['Lineal Simple', 'Lineal Múltiple', 'Polinómica (g=2)'],\n",
    "    'R²':            [r2_s,  r2_m,  r2_p],\n",
    "    'MAE (°C)':      [mae_s, mae_m, mae_p],\n",
    "    'RMSE (°C)':     [rmse_s, rmse_m, rmse_p],\n",
    "    'R² CV (prom)':  [cv_simple.mean(), cv_multi.mean(), cv_poly.mean()],\n",
    "    'R² CV (std)':   [cv_simple.std(),  cv_multi.std(),  cv_poly.std()],\n",
    "}).round(4)\n",
    "\n",
    "print('=' * 70)\n",
    "print('        COMPARACIÓN FINAL DE MODELOS — PREDICCIÓN temp_max')\n",
    "print('=' * 70)\n",
    "print(resultados.to_string(index=False))\n",
    "print('=' * 70)\n",
    "\n",
    "mejor = resultados.loc[resultados['R²'].idxmax(), 'Modelo']\n",
    "print(f'\\n🏆 Mejor modelo: {mejor} (R² = {resultados[\"R²\"].max():.4f})')\n",
    "\n",
    "# Tabla visual\n",
    "fig, ax = plt.subplots(figsize=(14, 4))\n",
    "ax.axis('off')\n",
    "fig.patch.set_facecolor('#1a1a2e')\n",
    "\n",
    "table = ax.table(\n",
    "    cellText=resultados.values,\n",
    "    colLabels=resultados.columns,\n",
    "    cellLoc='center', loc='center',\n",
    "    bbox=[0, 0, 1, 1]\n",
    ")\n",
    "table.auto_set_font_size(False)\n",
    "table.set_fontsize(11)\n",
    "\n",
    "for (row, col), cell in table.get_celld().items():\n",
    "    if row == 0:\n",
    "        cell.set_facecolor('#2d2d5e')\n",
    "        cell.set_text_props(color='white', fontweight='bold')\n",
    "    elif row % 2 == 0:\n",
    "        cell.set_facecolor('#1e1e3f')\n",
    "        cell.set_text_props(color='white')\n",
    "    else:\n",
    "        cell.set_facecolor('#16213e')\n",
    "        cell.set_text_props(color='white')\n",
    "    cell.set_edgecolor('#333366')\n",
    "\n",
    "ax.set_title('Tabla Comparativa de Métricas', color='white', fontsize=13, fontweight='bold', pad=20)\n",
    "plt.savefig('../data/ml_tabla_metricas.png', dpi=120, bbox_inches='tight', facecolor='#1a1a2e')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 11. Predicción por Ciudad"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Última medición por ciudad\n",
    "ultimo_registro = (\n",
    "    df.sort_values('fecha_consulta')\n",
    "      .groupby('ciudad')\n",
    "      .last()\n",
    "      .reset_index()\n",
    ")\n",
    "\n",
    "X_pred    = ultimo_registro[FEATURES_MULTIPLE].fillna(0).values\n",
    "X_pred_sc = scaler.transform(X_pred)\n",
    "\n",
    "ultimo_registro['temp_max_predicha'] = lr_multi.predict(X_pred_sc)\n",
    "ultimo_registro['error_estimado']    = (\n",
    "    (ultimo_registro['temp_max_predicha'] - ultimo_registro['temp_max']).abs()\n",
    ")\n",
    "\n",
    "resultado_ciudades = ultimo_registro[[\n",
    "    'ciudad', 'departamento', 'temperatura',\n",
    "    'temp_max', 'temp_max_predicha', 'error_estimado'\n",
    "]].sort_values('temp_max_predicha', ascending=False)\n",
    "\n",
    "print('🌡️  Predicción de Temperatura Máxima por Ciudad:')\n",
    "print(resultado_ciudades.round(2).to_string(index=False))\n",
    "\n",
    "# Gráfica\n",
    "fig, ax = plt.subplots(figsize=(14, 8))\n",
    "fig.patch.set_facecolor('#1a1a2e')\n",
    "ax.set_facecolor('#1a1a2e')\n",
    "\n",
    "x_pos = np.arange(len(resultado_ciudades))\n",
    "ax.bar(x_pos - 0.2, resultado_ciudades['temp_max'], 0.4,\n",
    "       label='Temp. Máx. Real', color=COLORS[0], alpha=0.8)\n",
    "ax.bar(x_pos + 0.2, resultado_ciudades['temp_max_predicha'], 0.4,\n",
    "       label='Temp. Máx. Predicha', color=COLORS[1], alpha=0.8)\n",
    "\n",
    "ax.set_xticks(x_pos)\n",
    "ax.set_xticklabels(resultado_ciudades['ciudad'], rotation=45, ha='right', color='white', fontsize=8)\n",
    "ax.set_ylabel('Temperatura (°C)', color='white')\n",
    "ax.set_title('Temperatura Máxima Real vs Predicha por Ciudad', color='white', fontsize=14, fontweight='bold')\n",
    "ax.legend()\n",
    "ax.grid(True, alpha=0.2, axis='y')\n",
    "ax.tick_params(colors='white')\n",
    "plt.tight_layout()\n",
    "plt.savefig('../data/ml_prediccion_ciudades.png', dpi=120, bbox_inches='tight', facecolor='#1a1a2e')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 12. Guardar Modelo y Scaler"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "os.makedirs('../ml', exist_ok=True)\n",
    "\n",
    "with open('../ml/modelo_regresion.pkl', 'wb') as f:\n",
    "    pickle.dump(lr_multi, f)\n",
    "\n",
    "with open('../ml/scaler.pkl', 'wb') as f:\n",
    "    pickle.dump(scaler, f)\n",
    "\n",
    "with open('../ml/features.pkl', 'wb') as f:\n",
    "    pickle.dump(FEATURES_MULTIPLE, f)\n",
    "\n",
    "# Guardar resultados\n",
    "resultado_ciudades.to_csv('../data/predicciones_ciudades.csv', index=False)\n",
    "resultados.to_csv('../data/metricas_modelos.csv', index=False)\n",
    "\n",
    "print('✅ Modelo guardado en:   ml/modelo_regresion.pkl')\n",
    "print('✅ Scaler guardado en:   ml/scaler.pkl')\n",
    "print('✅ Features guardados:   ml/features.pkl')\n",
    "print('✅ Predicciones CSV:     data/predicciones_ciudades.csv')\n",
    "print('✅ Métricas CSV:         data/metricas_modelos.csv')\n",
    "print(f'\\n🎯 Modelo final → R² = {r2_m:.4f} | RMSE = {rmse_m:.4f} °C | MAE = {mae_m:.4f} °C')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 13. Conclusiones\n",
    "\n",
    "### Hallazgos principales\n",
    "\n",
    "1. **Regresión Simple** — La temperatura actual explica gran parte de la variación en `temp_max` (alta correlación esperada). Modelo interpretable pero limitado a una sola variable.\n",
    "\n",
    "2. **Regresión Múltiple** — Incorporar humedad, presión, nubosidad, sensación térmica y variables temporales mejora significativamente el R² y reduce el error de predicción.\n",
    "\n",
    "3. **Regresión Polinómica** — Captura relaciones no lineales entre variables. Útil cuando los residuos del modelo múltiple muestran patrones curvos.\n",
    "\n",
    "4. **Variables más importantes**: la temperatura actual y la sensación térmica tienen los coeficientes más altos. La humedad tiene efecto negativo (a mayor humedad, menor `temp_max` típicamente).\n",
    "\n",
    "5. **Validación cruzada 10-fold** confirma que los modelos generalizan bien sin overfitting severo. El uso de `Pipeline` garantiza que no hay data leakage entre folds.\n",
    "\n",
    "### Recomendaciones\n",
    "\n",
    "- Acumular más datos históricos mejorará la precisión del modelo\n",
    "- Considerar modelos separados por región climática (costa, altiplano, Amazonía)\n",
    "- Explorar modelos de series de tiempo (ARIMA, Prophet) para tendencias de largo plazo\n",
    "- Integrar variables externas: elevación del terreno y estación del año"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.12.3"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}
